# Reproducibility and Inspection

Suppose we change one line of a training script and the validation accuracy
of a Fashion-MNIST classifier rises by 0.3 points. Before we credit the
change, we need to know how much two runs of the unchanged script differ.
For the classifiers of that section and
that section, they differ by a few tenths of a point, because
each run draws different initial weights and visits the examples in a
different order. A run that fails is a second problem of the same kind. If
the loss turns into NaN after hours of training, we want to repeat the run
exactly up to that point and then look inside the model to see which layer
produced the first non-finite number.

This section first makes runs repeatable. Every random draw comes from a
generator whose sequence of numbers is fixed by a seed, so seeding the
generators fixes the draws; repeating a run bit for bit also requires that
the arithmetic adds numbers in a fixed order. The section then looks inside
running models with *hooks*, functions that observe each layer of an
unmodified model as it computes. The two halves depend on each other: a
measurement taken inside a model can be compared across two versions of the
code only if the runs differ in nothing but that code.

In [1]:
import random
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

## Seeds and Random Number Generators

Randomness enters a training run in more places than one might list at
first: initialization draws every weight (that section), dropout
draws a new mask at every step, the data loader shuffles the examples
differently in every epoch, data augmentation draws crops and flips, and
parallel loader workers do some of this work in separate processes. Each
draw comes from a *pseudorandom number generator*, a deterministic algorithm
whose sequence of outputs is fixed by its starting state, the *seed*.
Seeding every generator that a run consults therefore fixes every draw.

### Global Seeds

Where the random functions get their state from is the first thing to know
about a framework: a global generator that they consult unless told
otherwise, or keys passed explicitly.

`torch.manual_seed(k)` seeds PyTorch's global generator on the CPU and on
every GPU. It does not seed Python's `random` module or NumPy's global
generator (`random.seed`, `np.random.seed`), nor NumPy generator objects
created with `np.random.default_rng`, which carry seeds of their own. The
function below draws everything, weights and data, from PyTorch's
generator, so one call fixes all its draws:

In [2]:
def train_once(seed):
    torch.manual_seed(seed)  # seeds the CPU and every GPU
    net = nn.Sequential(nn.Linear(20, 32), nn.ReLU(), nn.Linear(32, 1))
    opt = torch.optim.SGD(net.parameters(), lr=0.1)
    X, y = torch.randn(128, 20), torch.randn(128, 1)
    for _ in range(5):
        opt.zero_grad()
        loss = ((net(X) - y) ** 2).mean()
        loss.backward()
        opt.step()
    return loss.item(), net[0].weight.detach().clone()

loss_a, w_a = train_once(seed=0)
loss_b, w_b = train_once(seed=0)
loss_c, _ = train_once(seed=1)
print('same seed, identical loss and weights:',
      loss_a == loss_b, torch.equal(w_a, w_b))
print('different seed:', loss_a, 'vs', loss_c)

same seed, identical loss and weights: True True
different seed: 0.8643209338188171 vs 0.9312677383422852


The two runs with seed 0 agree *bitwise*, down to the last bit of the loss
and of every weight: same initial weights, same data, same gradients, same
operations in the same order. The run with seed 1 ends with a different
loss.


### Generator Objects

A single global generator couples all the code that draws from it. Inserting
one extra random call, a new layer's initialization or a stray draw while
debugging, shifts every number drawn after it, so the data split, the
initialization, and the dropout masks all change at once. A *generator
object* is a private stream with its own seed, which unrelated draws cannot
shift.

A `torch.Generator` is such a stream. Here a random permutation, say a split
of a dataset into training and validation examples, stays fixed no matter
what else draws from the global generator in between:

In [3]:
g = torch.Generator().manual_seed(42)
split = torch.randperm(10, generator=g)
_ = torch.randn(1000)  # unrelated draws from the global generator
split_again = torch.randperm(10, generator=g.manual_seed(42))
print(torch.equal(split, split_again), split)

True tensor([2, 6, 1, 8, 4, 5, 0, 9, 3, 7])


Most sampling functions accept `generator=`, and so does `DataLoader`, where
it sets the shuffle order. We use that next.

### Data Loader Workers

Data loaders often prepare minibatches in parallel *worker* processes. A
worker that draws random numbers, for data augmentation for instance, needs
a stream of its own. Two failures are common. If every worker starts as a copy of the
main process, as with the *fork* start method, Linux's default before
Python 3.14, the workers inherit the same generator
state and apply the same "random" augmentations. If each worker seeds itself
from the operating system's entropy, no two runs agree.

`DataLoader` gives every worker its own PyTorch seed, derived from a base
seed and the worker's index, and seeds Python's `random` module and NumPy's
global generator in each worker from it. It cannot reach generator objects
that a dataset or a library creates for itself; those need a *worker
initializer*, a function that the loader calls once in each new worker. The
loader's `generator` sets both the base seed and the shuffle order, so one
generator fixes the whole input pipeline. The initializer below follows
PyTorch's reproducibility notes. For `random` and NumPy it repeats what the
loader already does; it shows where a dataset's own generator objects would
be replaced:

In [4]:
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32  # this worker's torch seed
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator().manual_seed(0)
data = TensorDataset(torch.arange(8))
# num_workers=0 keeps this cell portable; with num_workers > 0, each
# worker process calls seed_worker once when it starts
loader = DataLoader(data, batch_size=4, shuffle=True, num_workers=0,
                    worker_init_fn=seed_worker, generator=g)
print([batch[0].tolist() for batch in loader])
g.manual_seed(0)
print([batch[0].tolist() for batch in loader])

[[5, 3, 1, 7], [2, 6, 4, 0]]
[[5, 3, 1, 7], [2, 6, 4, 0]]


Seeding the generator again reproduces the shuffle order.

### Randomness as a Value

The worker failures above arise because the state of a generator is copied
or shared without appearing in the program's data flow. JAX makes randomness
a value instead: every random function takes a key, the same key gives the
same numbers, and independent streams come from splitting a key. A key can
still be reused by mistake, but the reuse is visible in the code. The cost
is that every function that draws random numbers must accept a key and pass
split keys on. A global seed is the convenient form of the same contract: one
implicit key that all code shares. Generator objects recover the explicit
form where it matters.

PyTorch offers both forms: the global generator, and `torch.Generator`
objects passed to the functions that should use them.

## Deterministic Arithmetic

Seeding fixes which numbers the program draws. It does not fix how the
arithmetic combines them. Floating-point addition is not associative
(that section): every sum is rounded to the nearest representable
number, so adding the same numbers in a different order can give a different
result.

The smallest example takes three float32 numbers. Near $10^8$, adjacent
float32 numbers are 8 apart, so $-10^8 + 1$ rounds back to $-10^8$:

In [5]:
a, b, c = [torch.tensor(v, dtype=torch.float32) for v in (1e8, -1e8, 1.0)]
print(float((a + b) + c), float(a + (b + c)))  # same numbers, regrouped

1.0 0.0


Regrouping changes the result from 1 to 0. In a long sum the effect is
smaller but of the same kind: adding a million random numbers forward and
backward usually gives results that differ in their last bits.

On a CPU, the order in which one operation adds its numbers is usually fixed
for a given number of threads, so a seeded run repeats. On a GPU, many
threads add partial sums at the same time, and some operations combine them
with *atomic additions*, which commit each partial sum in whatever order the
threads happen to finish. Examples are the gradients of an embedding lookup
or of a scatter operation, some convolution gradients, and some reductions.
Two seeded runs on the same machine can then differ in the last bits of
their weights, and over many training steps such differences grow until the
loss curves differ visibly. The remedy is a switch that replaces such
operations with deterministic ones, which add in a fixed order and are often
slower.

`torch.use_deterministic_algorithms(True)` is the switch. Operations that
have a deterministic implementation use it; operations without one raise a
`RuntimeError` instead of returning nondeterministic results. On a GPU, the
environment variable `CUBLAS_WORKSPACE_CONFIG=:4096:8` must also be set, or
matrix multiplications raise as well. The backward pass of 3-D average
pooling on a GPU has no deterministic implementation:

In [6]:
torch.use_deterministic_algorithms(True)
if torch.cuda.is_available():
    try:
        x = torch.randn(1, 1, 2, 2, 2, requires_grad=True, device='cuda')
        nn.AvgPool3d(1)(x).sum().backward()
    except RuntimeError as e:
        print(str(e).split(', but')[0])
else:
    print('CPU run: every kernel used here is deterministic;',
          'on a GPU, operations without a deterministic kernel raise')
torch.use_deterministic_algorithms(False)

avg_pool3d_backward_cuda does not have a deterministic implementation


Bitwise agreement holds only for one machine, one library version, and one
configuration of these switches; no framework promises it across releases,
across hardware, or between CPU and GPU. Bitwise reproducibility is
therefore a tool for debugging: it lets us find the step at which two runs
first differ. The scientific goal is weaker and more robust. A conclusion,
such as "the change raises accuracy", should hold across seeds, so we report
the mean and the spread over several runs rather than one run. As with the
number formats of that section, a conclusion that changes with the
seed or with the last bits of the arithmetic needs more evidence.

## Model Inspection with Hooks

A repeatable run tells us that a problem will recur, not where it arises.
To find out, we observe what each layer computes during the run. Editing the
`forward` method of every layer to print or store its output is tedious, and
impossible for a model that came from a library or a checkpoint whose code
we do not want to change. A *hook* is a function that the framework calls at
a fixed point of each layer's computation, from outside the layer's code.

In that section we saw that `net(X)` does not call
`forward` directly: it calls `__call__`, which wraps `forward`. Hooks run in
that wrapper. `module.register_forward_hook(f)` arranges for
`f(module, inputs, output)` to run after every forward pass of the module,
and `register_forward_pre_hook` before it. the figure shows where
they run. A forward hook that returns a value replaces the module's output.

![The `__call__` wrapper as a pipeline: input flows through pre-hooks, then forward, then hooks, to the output, with the two hook stages dashed and orange against forward's solid blue, and a side arrow from the hooks stage to an observer that can capture, check, or modify.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-hooks.svg)

As a test model we use a stack of eight residual blocks of width 64 between
an input layer with 20 inputs and an output layer with 10 outputs. The block
is the residual block of that section, unchanged: it adds
the output of a small network, its *body*, to its input: $\mathbf{h} = \mathbf{x} +
\textrm{body}(\mathbf{x})$, where the body is a linear layer, a ReLU, and a
second linear layer. The weights start from Xavier initialization
(that section) and the biases at zero.

In [7]:
torch.seed()  # a fresh random seed: the demos above left one pinned

class ResidualBlock(nn.Module):
    def __init__(self, num_hiddens):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(num_hiddens, num_hiddens),
                                  nn.ReLU(),
                                  nn.Linear(num_hiddens, num_hiddens))

    def forward(self, X):
        return X + self.body(X)

net = nn.Sequential(nn.Linear(20, 64),
                    *[ResidualBlock(64) for _ in range(8)],
                    nn.Linear(64, 10))
for m in net.modules():
    if isinstance(m, nn.Linear):  # Xavier weights, zero biases
        nn.init.xavier_uniform_(m.weight)
        nn.init.zeros_(m.bias)

### Activation Statistics

The initialization experiments of that section measured the
standard deviation of the activations at every depth. With hooks, the same
measurement takes a few lines on an unmodified model. We record the standard
deviation of each layer's output on a minibatch of 256 standard normal
inputs.

In [8]:
stats, handles = [], []

def record(module, inputs, output):
    stats.append((type(module).__name__, output.detach().std().item()))

for m in net:
    handles.append(m.register_forward_hook(record))
net(torch.randn(256, 20))
for h in handles:
    h.remove()
for name, std in stats:
    print(f'{name:15s} std {std:.2f}')

Linear          std 0.69
ResidualBlock   std 0.85
ResidualBlock   std 1.06
ResidualBlock   std 1.34
ResidualBlock   std 1.61
ResidualBlock   std 1.95
ResidualBlock   std 2.47
ResidualBlock   std 3.18
ResidualBlock   std 4.17
Linear          std 4.53


The standard deviation grows from block to block, from about 0.7 after the
input layer to roughly 3 to 4 after the eighth block. Each block adds its
body's output to its input, and the two are nearly uncorrelated, so their
variances add. With Xavier initialization the body's output has about half
the variance of its input: the first linear layer keeps the variance, the
ReLU halves the mean square, and the second linear layer keeps it again. The
variance therefore grows by a factor of about 1.5 per block, and the
standard deviation by about $\sqrt{1.5} \approx 1.22$. This growth with
depth is why the residual branches of deep networks are often initialized
with smaller weights (that section).

Two rules keep hooks safe. First, *detach before storing*: the hook above
stores `output.detach().std()`. Storing `output` itself would keep the
autograd graph of every forward pass in memory until `stats` is cleared, a
leak that grows with every minibatch. Second, *keep the handle and remove
it*: a hook stays registered for the module's lifetime unless we call
`handle.remove()`, slowing every later forward pass and keeping alive any
tensors it stores.

### A NaN Finder

When a loss becomes NaN at step 40,000, the useful question is which layer
produced the first non-finite value. A hook on every layer that checks its
output for infinities and NaNs answers it in one forward pass: the first
hook to fire names the layer, because hooks fire in the order the layers
compute. To test this, we put a NaN into one weight of the fourth layer of
the stack, the third residual block:

In [9]:
def make_finite_check(name):
    def check(module, inputs, output):
        if not torch.isfinite(output).all():
            raise RuntimeError(f'first non-finite output in {name}')
    return check

handles = [m.register_forward_hook(make_finite_check(name))
           for name, m in net.named_modules()
           if len(list(m.children())) == 0]  # leaf modules only
saved = net[3].body[0].weight[0, 0].item()
with torch.no_grad():
    net[3].body[0].weight[0, 0] = float('nan')  # break one layer
try:
    net(torch.randn(2, 20))
except RuntimeError as e:
    print(e)
for h in handles:
    h.remove()
with torch.no_grad():
    net[3].body[0].weight[0, 0] = saved  # restore the weight

first non-finite output in 3.body.0


The report names `3.body.0`, the layer that holds the NaN, without a single
print statement in the model.

### Gradient Inspection

Gradients can be inspected in the same way, for instance to log the size of
each layer's gradient or to find where the gradients start to explode. As a
loss we take the mean squared output of the stack on a minibatch of 256
standard normal inputs.

`module.register_full_backward_hook(f)` runs `f(module, grad_input,
grad_output)` while the backward pass runs, once the gradients with respect
to the module's inputs are computed. Both arguments are tuples with one
gradient per input or output of the module, so the gradient with respect to
a block's output is `grad_output[0]`. A hook that returns a value replaces
`grad_input`. Like a forward hook, a backward hook returns a handle to
remove and should detach anything it stores. The gradients of the
parameters themselves are available after the backward pass as `p.grad`.
Here are the norms of the gradients with respect to the output of each
residual block, recorded by hooks, and the gradient norms of the first
linear layer of the third residual block:

In [10]:
X = torch.randn(256, 20)
grad_norms, handles = [], []
def record(module, grad_input, grad_output):
    grad_norms.append(grad_output[0].norm().item())  # store a number
for block in net[1:-1]:
    handles.append(block.register_full_backward_hook(record))
net.zero_grad()
(net(X) ** 2).mean().backward()
for h in handles:
    h.remove()
# The hooks fire in backward order, last block first
print('block output gradients:',
      ' '.join(f'{g:.3g}' for g in reversed(grad_norms)))
layer = net[3].body[0]  # block 3, first layer
print(f'weight {layer.weight.grad.norm().item():.3g}, '
      f'bias {layer.bias.grad.norm().item():.3g}')

block output gradients: 1.51 1.22 0.948 0.724 0.558 0.42 0.325 0.253
weight 21.1, bias 4.43


We can now repeat a run bit for bit on one machine, say which sources of
randomness and which operations stand in the way, and observe any layer of
an unmodified model during a run. What these tools cannot do is make two
different machines agree, or decide on their own whether a difference
between two runs matters; for that we compare results across seeds.

## Summary

Every random draw in a training run comes from a generator fixed by a seed:
initialization, dropout, shuffling, augmentation, and loader workers all
consult one. Seeding each generator the run uses fixes the draws, and private
generator objects or explicit keys keep unrelated draws from shifting them.
Seeding does not fix the arithmetic: floating-point addition depends on the
order of the terms, and some GPU operations add in a varying order, so
bitwise repetition also needs deterministic operations and holds only on one
machine and one software version. It is a tool for debugging; scientific
conclusions should hold across seeds. Hooks, or their equivalents in each
framework, observe the outputs and gradients of every layer of an unmodified
model, which turns questions such as "where did the first NaN appear?" into
one forward pass.

## Exercises

1. [code] **Block ablation.** A forward hook that returns a value
   *replaces* the module's output.
    1. Use such a hook to zero the output of one residual block's body,
       which turns the block into the identity, and measure the relative
       change in `net`'s output on a fixed batch. Repeat for every block,
       and rank the blocks by this change.
    1. Rebuild `net` three more times with new random weights and repeat
       the ranking. State whether the conclusion "block $k$ matters most"
       survives, and what evidence such a claim would need.
2. [code] **Residual variance budget.** Let $\mathbf{x}_k$,
   $\mathbf{b}_k$ and $\mathbf{h}_k$ denote the input, body output and
   output of block $k$, with each variance taken over all entries. On one
   batch, record for every `ResidualBlock` the variance of $\mathbf{x}_k$
   with `register_forward_pre_hook` (the pre-hook receives the argument
   tuple, so unpack it), of $\mathbf{h}_k$ with a forward hook on the
   block, and of $\mathbf{b}_k$ with a forward hook on its `body`. The
   standard deviations in the table of that section
   grow because each block adds its body's output to the stream. Measure
   how closely

   $$\textrm{Var}[\mathbf{h}_k] = \textrm{Var}[\mathbf{x}_k] + \textrm{Var}[\mathbf{b}_k]$$

   holds for every block, and name the term this equation omits.
3. [code] **Seeds and data loading.** ● Extend `train_once` to load its
   data through a `DataLoader` with `shuffle=True` and `num_workers=4`,
   keeping the single `torch.manual_seed(seed)` call at its top. The
   dataset adds noise to each example inside `__getitem__`, drawn from an
   `np.random.default_rng()` object created in the dataset's constructor.
   Run the code as a standalone script, because worker processes started
   with `spawn`, the default on macOS and Windows, cannot import a dataset
   class defined in a notebook.
    1. Predict whether two runs with the same seed agree on the shuffle
       order and on the noise, and whether the four workers draw the same
       noise. Justify each prediction from where that randomness is drawn,
       then verify all three.
    1. Extend `seed_worker` to replace the dataset's generator with one
       derived from the worker's own seed, and verify that two runs now
       agree and that the workers draw different noise.

    *Adapted from the PyTorch documentation
    [Multi-process data loading](https://docs.pytorch.org/docs/stable/data.html#randomness-in-multi-process-data-loading).*
4. [code] **Divergence monitor.** ● Turn the finiteness check of
   that section into a per-step training monitor. Rebuild
   `net` and train it with SGD at learning rate 1 on the mean squared
   error between its output and standard normal targets, using one fixed
   batch of 256 standard normal inputs. Report the first step at which a
   layer's output is non-finite, and the first such layer by depth. Check
   the loss at every step as well, and report whether it becomes
   non-finite before any layer output does.
5. [code] **Gradient norms.** ● Rebuild `net`, multiply the weight of
   every linear layer inside its residual blocks by 3, and use the mean
   squared output on a fixed batch as the loss.
    1. Using `register_full_backward_hook`, record the norm of each block's
       output gradient. After the backward pass, compute each block's
       parameter-gradient norm, and plot both families against depth.
    1. Express the gradient of a linear layer's weight through the layer's
       input and its output gradient. Use this expression to explain why
       the two depth profiles differ, and state which family to monitor for
       an exploding backward signal.
6. [code] **Hook coverage.** Attach a forward hook to every leaf module of
   `torchvision.models.resnet18()` through `named_modules()`, and count
   how often each hook fires in one forward pass. Explain any module that
   fires more than once, and name a computation in `BasicBlock.forward`
   that no module hook observes.

[Discussions](https://d2l.discourse.group/t/36759)